# Saliency：预计算（主线）+ 诊断（方法对比 / 重建对比 / 渲染检查 / checkpoint 指标）

对应 `saliency_map/scripts/precompute_saliency_maps.py`（主线，产出训练用的 `saliency_occlusion.npz`）
和 `saliency_map/scripts/diagnostics/` 下 4 个脚本（`compare_heatmap_methods.py` / `compare_recon.py` /
`preview_cartpole_render.py` / `eval_decoder.py`）。

**运行前**：Jupyter kernel 切换成 `starv_shared`（`/home/tealab_shared/starv/env/starv_shared`）。
本文件放在 `verifiable_wm/notebooks/` 下，第一个 cell 会自动定位仓库根目录并切换过去。

In [ ]:
import os
os.environ.setdefault("PYGLET_HEADLESS", "True")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-codex")

import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "dynamic.py").exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR.parent
assert (NOTEBOOK_DIR / "dynamic.py").exists(), (
    f"没找到仓库根目录（dynamic.py），请从 verifiable_wm/ 或 verifiable_wm/notebooks/ 启动 Jupyter"
)
os.chdir(NOTEBOOK_DIR)
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

print("repo root:", NOTEBOOK_DIR)

In [ ]:
# saliency_map/scripts/ 和 saliency_map/scripts/diagnostics/ 不是顶层包，
# 直接把这两个目录加进 sys.path，按文件名 import（跟脚本自己 REPO_ROOT 的 __file__ 解析互不冲突）。
sys.path.insert(0, str(NOTEBOOK_DIR / "saliency_map" / "scripts"))
sys.path.insert(0, str(NOTEBOOK_DIR / "saliency_map" / "scripts" / "diagnostics"))

import precompute_saliency_maps as psm
import compare_heatmap_methods as chm
import compare_recon as cr
import preview_cartpole_render as pcr
import eval_decoder as ed

## 0. 预计算 saliency map（主线，`precompute_saliency_maps.py`）

给 decoder 训练数据算 controller occlusion heatmap，写回
`datasets/<env>/data/dataset_v1/saliency_occlusion.npz`，供 `train_decoder.py` 的 `saliency`/`hybrid`
权重模式使用。已经生成过、只是想重新看看诊断图的话不需要重跑这一节。

In [ ]:
psm.run(psm.parse_args([
    "--config", "config/make_decoder_dataset/cartpole.json",
]))

In [ ]:
psm.run(psm.parse_args([
    "--config", "config/make_decoder_dataset/pendulum.json",
]))

## 1. Saliency 方法对比（`compare_heatmap_methods.py`）

把 vanilla grad / SmoothGrad / SmoothGrad² / IG-white² / Grad-CAM / Occlusion-white 六种方法
画在同一张图里，纯视觉对比，不代表任何一个是正式结论。

In [ ]:
chm.run(chm.parse_args([
    "--config", "config/make_decoder_dataset/cartpole.json",
]))

In [ ]:
chm.run(chm.parse_args([
    "--config", "config/make_decoder_dataset/pendulum.json",
]))

## 2. Decoder 重建对比（`compare_recon.py`）

同一批 test state，多个 decoder checkpoint 重建出的图像并排对比真实图像。
默认 checkpoint 组合见 `compare_recon.DEFAULT_CHECKPOINTS`；要换 checkpoint 直接传自定义 dict。

In [ ]:
cr.run(
    env_name="cartpole",
    dataset_dir=cr.default_dataset_dir("cartpole"),
    checkpoints=cr.DEFAULT_CHECKPOINTS["cartpole"],
)

In [ ]:
cr.run(
    env_name="pendulum",
    dataset_dir=cr.default_dataset_dir("pendulum"),
    checkpoints=cr.DEFAULT_CHECKPOINTS["pendulum"],
)

## 3. CartPole 渲染健全性检查（`preview_cartpole_render.py`）

对比 dataset 里保存的图像和用当前 `env.py`/`dynamic.py` 重新渲染的图像，
防止再出现旧的黑块渲染 bug。数据集已经修好后不必每次都跑，回归检查时用。

In [ ]:
pcr.run(pcr.parse_args(["--render-current"]))

## 4. 任意 checkpoint 指标读数（`eval_decoder.py`）

对不是 `train_decoder.py` 直接训出来的 checkpoint（比如 raw/old decoder）算 test 指标，
复用 `train_decoder.py` 的 `load_split`/`compute_weight`/`evaluate`，跟 `metrics.json` 里的数字算法一致。
改 `--config`/`--weights` 换成想看的 checkpoint。

In [ ]:
ed.run(ed.parse_args([
    "--config", "config/train_decoder/pendulum/intensity.json",
    "--weights", "dwm_weight/raw_weight/pendulum/decoder_pen.pth",
    "--label", "raw (old decoder)",
]))